# 14 RHCR：滚动时域消冲突


本课按百科条目写。RHCR 把终身/在线 MAPF 拆成一串窗口规划。调度台在最后。

---

## 1. 定义

**Rolling-Horizon Collision Resolution**（Li et al., AAAI 2021）：执行层只走一小段，规划层反复对未来 w 拍消冲突。

```
每 h 拍（本课 h=1）:
    以当前真实位置为起点
    只对未来 w 拍消冲突（WHCA / CBS / PBS 均可）
    执行第 1 步（或下发 ≤ w 的段）
```

本仓库 `rhcr` 窗口规划用 WHCA，每轮只前进一步。

终身 MAPF：任务中途出现、目标会变。一次 HCA 到终点的假设不成立，滚动是默认架构。

---

## 2. 保证

无完备、无最优。窗口短则快、短视（可能把车引进死胡同）；窗口长则接近一次性规划、更慢。目标中途改变时不必作废整条路。

执行层：未锁的段不能走。评测看 `max_issue ≤ window`。

---

## 3. 不变量

- 每轮 `pos` 是模拟器里的当前格。
- 下发段长度 ≤ w。
- 到齐：所有 `pos[a]==goals[a]` 则停。
- 每轮 WHCA 的占用表应是 **这一窗的车**，不要把上轮旧锁留着（本实现每轮新建 WHCA，表是新的）。

---

## 4. 伪代码（本仓库）

```
pos ← starts
重复 steps 次或全部到达:
    plan ← WHCA(pos, goals, window=w)
    对每车: 执行 plan 的第 1 步，更新 pos
    记录 issued 段
```

---

## 5. 手算

两车平行、目标 5 步、w=3。第 1 轮 WHCA 给出 3 步前缀，只执行第 1 步。第 2 轮从新位置再规划 3 步。约 5 轮到齐。`max_issue` 是某轮下发的最长段，教学实现 `seg[:window]` 再只走 1 步，issued 长度仍 ≤ w。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：滚动循环


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.algos.prioritized import whca


def rhcr_simple(graph, starts, goals, window=3, steps=20):
    pos = dict(starts)
    full_paths = {a: [starts[a]] for a in starts}
    issued_log = []
    for step in range(steps):
        if all(pos[a] == goals[a] for a in starts):
            break
        plan = whca(graph, pos, goals, window=window)
        issued = {}
        for a, path in plan["paths"].items():
            seg = path[:window]
            issued[a] = seg
            if len(seg) > 1:
                pos[a] = seg[1]
                full_paths[a].append(pos[a])
        issued_log.append(issued)
        print(f"t={step} pos={pos} issue_len={ {a: len(s) for a, s in issued.items()} }")
    found = all(full_paths[a][-1] == goals[a] for a in starts)
    max_issue = max((len(seg) for batch in issued_log for seg in batch.values()), default=0)
    return {"found": found, "paths": full_paths, "max_issue": max_issue, "window": window}


g, starts, goals = two_agent_tunnel()
r = rhcr_simple(g, starts, goals, window=3, steps=20)
print("到齐", r["found"], "max_issue", r["max_issue"], "应<=", r["window"])
print("轨迹 a", r["paths"]["a"])
print("轨迹 b", r["paths"]["b"])


## 7. 逐行

| 行 | 作用 |
|----|------|
| `pos` | 模拟器当前格，当下一轮 WHCA 的起点 |
| `seg[1]` | 每轮只前进一步（h=1） |
| `max_issue` | eval：不得超过 window |
| 每轮新 WHCA | 占用表不跨窗残留 |

库 `rhcr` 同样结构。窗口规划可换成 CBS（论文常见），教学用 WHCA 更快。


In [ ]:
from lib.algos.prioritized import rhcr

lib_r = rhcr(g, starts, goals, window=3, steps=20)
print("库 到齐", lib_r["found"], "max_issue", lib_r["max_issue"])
print("轨迹 a", lib_r["paths"]["a"])


## 常见 bug

1. 每轮 WHCA 的 table 不空：旧锁把当前车自己锁死。
2. 执行了窗口内全部点却不重规划，失去滚动意义。
3. 用「单次 WHCA 是否到终点」当 RHCR 成功标准。
4. window=1 且两车必须换位：一步窗口可能永远对顶，要加大 w 或换 PBS/CBS 当窗规划器。

## 条目小结

| 项目 | 内容 |
|------|------|
| 架构 | 规划窗 + 执行 1 步 + 重规划 |
| 保证 | 无 |
| 用途 | 终身 / 在线 / 目标会变 |
| 下一课 | 要 SOC 最优 → CBS 冲突树 |

## 练习

1. w=1 与 w=5 到齐轮数、短视程度如何变？
2. 执行 h=w（把窗内全走完再规划）和 h=1 各有什么风险？
3. 新任务在 t=3 出现，RHCR 要改哪一行？HCA 一次规划为什么麻烦？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), lib_r, "RHCR")
studio
